# InfraScan Database Schema

This notebook proposes a practical database schema for **InfraScan - AI-Powered Crack & Damage Assessment for Civil Infrastructure**.

The schema is a design proposal for planned application features. It does not create a database or claim that these tables are currently implemented.

## 1. Database overview and purpose

The database will store structured information about users, infrastructure assets, uploaded images, inspections, AI assessment results, reports, and security-relevant actions. Image bytes and generated overlays should normally be stored in object storage; the database stores their paths or object keys and searchable metadata.

The design should support the planned workflow from image upload through inspection review and reporting. It should also preserve enough history to identify who performed an action, which asset was inspected, which model produced a result, and when each event occurred.

## 2. Recommended database technology

**Recommended technology: PostgreSQL.**

PostgreSQL is a practical choice because it provides:

- Strong relational constraints for users, assets, inspections, images, results, and reports.
- Transactions so related records can be created or updated consistently.
- Useful indexes for filtering by user, asset, date, and processing status.
- Mature backup, migration, access-control, and monitoring tools.
- Optional support for JSON fields when model metadata changes, without making the core schema unstructured.

A local PostgreSQL instance or a managed PostgreSQL service can be used during future implementation. This notebook does not create or connect to a database.

## 3. Users table

The `users` table represents people who can access the planned application. Passwords must never be stored as plain text. If InfraScan manages passwords directly, `password_hash` should contain a strong adaptive hash such as Argon2id or bcrypt. An external identity provider could be used instead, in which case the table can store an external subject identifier and may not need a local password hash.

| Column | Suggested type | Constraints | Purpose |
| --- | --- | --- | --- |
| `user_id` | UUID | Primary key, generated | Unique user identifier. |
| `name` | VARCHAR(150) | NOT NULL | Display name. |
| `email` | VARCHAR(320) | NOT NULL, UNIQUE | Login and contact identifier. Store normalized form for uniqueness. |
| `password_hash` | TEXT | Nullable when using external identity | Password hash only; never the original password. |
| `role` | VARCHAR(40) or enum | NOT NULL | Planned role such as inspector, engineer, maintenance, or admin. |
| `created_at` | TIMESTAMPTZ | NOT NULL, default current time | Account creation time. |
| `updated_at` | TIMESTAMPTZ | NOT NULL, updated by application | Last profile or role update. |

A production design may use separate `roles` and `user_roles` tables if users can have multiple roles. For a beginner-friendly first version, one constrained role column is sufficient if the requirement is one role per user.

## 4. Infrastructure assets table

The `infrastructure_assets` table stores the concrete structure being inspected. Keeping asset information separate from images and inspections prevents repeating the same location and description for every upload.

| Column | Suggested type | Constraints | Purpose |
| --- | --- | --- | --- |
| `asset_id` | UUID | Primary key, generated | Unique asset identifier. |
| `asset_name` | VARCHAR(200) | NOT NULL | Human-readable name. |
| `infrastructure_type` | VARCHAR(60) | NOT NULL | Building, bridge, road, tunnel, wall, or another controlled value. |
| `location` | TEXT | NOT NULL | Address, site description, or future geographic reference. |
| `description` | TEXT | Nullable | Additional construction or maintenance context. |
| `created_at` | TIMESTAMPTZ | NOT NULL, default current time | Record creation time. |
| `updated_at` | TIMESTAMPTZ | NOT NULL, updated by application | Last asset update. |

The application should validate `infrastructure_type` against an allowed list or reference table. If geographic search is required later, latitude and longitude can be added with appropriate validation.

## 5. Image records table

The `image_records` table stores metadata for uploaded images. The actual image file should be held in private object storage or another controlled file store, not placed directly in a regular database text column.

| Column | Suggested type | Constraints | Purpose |
| --- | --- | --- | --- |
| `image_id` | UUID | Primary key, generated | Unique image identifier. |
| `asset_id` | UUID | NOT NULL, foreign key to `infrastructure_assets` | Asset shown in the image. |
| `uploaded_by` | UUID | NOT NULL, foreign key to `users` | User who uploaded the image. |
| `file_path` | TEXT | NOT NULL | Private object-storage key or controlled path. |
| `upload_date` | TIMESTAMPTZ | NOT NULL, default current time | Upload time. |
| `file_format` | VARCHAR(20) | NOT NULL | Detected format such as JPEG or PNG. |
| `image_status` | VARCHAR(30) | NOT NULL, constrained | Pending, validated, rejected, processing, or archived. |
| `file_size_bytes` | BIGINT | CHECK greater than zero | Size used for limits and monitoring. |
| `checksum` | VARCHAR(128) | Nullable, indexed | Helps detect duplicate or altered files. |
| `width` / `height` | INTEGER | CHECK greater than zero | Validated image dimensions. |

The stored path should use a generated identifier rather than trusting a user-provided filename. Original files should be treated as read-only evidence after validation.

## 6. Inspection records table

The `inspection_records` table groups one inspection activity for an asset. An inspection can contain multiple images and multiple AI assessment results.

| Column | Suggested type | Constraints | Purpose |
| --- | --- | --- | --- |
| `inspection_id` | UUID | Primary key, generated | Unique inspection identifier. |
| `asset_id` | UUID | NOT NULL, foreign key to `infrastructure_assets` | Asset being inspected. |
| `inspector_id` | UUID | NOT NULL, foreign key to `users` | User responsible for the inspection record. |
| `inspection_date` | TIMESTAMPTZ | NOT NULL | Date and time of the inspection. |
| `status` | VARCHAR(30) | NOT NULL, constrained | Draft, in progress, under review, completed, or cancelled. |
| `notes` | TEXT | Nullable | Human-entered context or review notes. |
| `created_at` | TIMESTAMPTZ | NOT NULL, default current time | Record creation time. |

A future association table can be added if one image may belong to several inspections. For a simple first implementation, an `inspection_id` foreign key on `image_records` is another reasonable option.

## 7. AI assessment results table

The `ai_assessment_results` table records an output produced by a particular model for an image. Storing model identity and assessment time is important because models may be retrained and produce different results later.

| Column | Suggested type | Constraints | Purpose |
| --- | --- | --- | --- |
| `result_id` | UUID | Primary key, generated | Unique result identifier. |
| `image_id` | UUID | NOT NULL, foreign key to `image_records` | Image analyzed. |
| `model_name` | VARCHAR(120) | NOT NULL | Human-readable model name. |
| `model_version` | VARCHAR(60) | NOT NULL | Exact model version or artifact identifier. |
| `predicted_defect_class` | VARCHAR(80) | NOT NULL | For example, negative, crack, or another validated class. |
| `confidence_score` | NUMERIC(6,5) | CHECK from 0 to 1 | Model confidence or calibrated score. |
| `assessment_date` | TIMESTAMPTZ | NOT NULL, default current time | Time inference completed. |
| `task_type` | VARCHAR(30) | NOT NULL | Classification, detection, or segmentation. |
| `result_path` | TEXT | Nullable | Object-storage key for an overlay, mask, or detailed JSON result. |
| `review_status` | VARCHAR(30) | NOT NULL, constrained | Pending review, accepted, corrected, or rejected. |
| `metadata` | JSONB | Nullable | Carefully controlled model-specific details. |

A confidence score is not a guarantee of correctness. The application should show uncertainty and allow qualified users to review or correct results. Detection boxes and segmentation masks can be stored in a structured result object referenced by `result_path` or in normalized child tables if detailed querying is required.

## 8. Reports table

The `reports` table tracks a generated summary for an inspection. The report file itself can be stored in private object storage.

| Column | Suggested type | Constraints | Purpose |
| --- | --- | --- | --- |
| `report_id` | UUID | Primary key, generated | Unique report identifier. |
| `inspection_id` | UUID | NOT NULL, foreign key to `inspection_records` | Inspection summarized by the report. |
| `created_by` | UUID | NOT NULL, foreign key to `users` | User who requested or generated the report. |
| `report_path` | TEXT | NOT NULL | Private object-storage key or controlled path. |
| `creation_date` | TIMESTAMPTZ | NOT NULL, default current time | Report creation time. |
| `report_status` | VARCHAR(30) | NOT NULL, constrained | Generating, ready, failed, or archived. |
| `report_version` | INTEGER | NOT NULL, default 1 | Allows report format changes to be tracked. |

Reports should identify the model versions and data used to create them. A report should not hide failed analyses or present an automated prediction as a confirmed engineering conclusion.

## 9. Audit log table

The `audit_logs` table records important user and system actions. It supports accountability, troubleshooting, and security review without storing unnecessary sensitive content.

| Column | Suggested type | Constraints | Purpose |
| --- | --- | --- | --- |
| `audit_id` | UUID | Primary key, generated | Unique event identifier. |
| `user_id` | UUID | Nullable, foreign key to `users` | User who performed the action; null for an automated system event. |
| `action` | VARCHAR(100) | NOT NULL | For example, upload_created, result_reviewed, or report_downloaded. |
| `entity_type` | VARCHAR(60) | NOT NULL | Type of affected record. |
| `entity_id` | UUID | Nullable | Identifier of the affected record. |
| `details` | JSONB | Nullable | Minimal structured context, excluding secrets. |
| `created_at` | TIMESTAMPTZ | NOT NULL, default current time | Action timestamp. |
| `ip_address` | INET | Nullable and access-controlled | Optional security investigation data. |

Audit rows should be append-only for normal application users. Retention and access to audit data should be defined according to organizational and legal requirements.

## 10. Primary keys, foreign keys, and relationships

### Primary keys

Each table has a primary key that uniquely identifies a row. UUIDs are recommended because they avoid exposing sequential counts and work well when records are created by more than one service. Auto-incrementing integers are also valid for a small internal deployment if the trade-off is understood.

### Foreign keys

Foreign keys preserve valid relationships:

- `image_records.asset_id` references `infrastructure_assets.asset_id`.
- `image_records.uploaded_by` references `users.user_id`.
- `inspection_records.asset_id` references `infrastructure_assets.asset_id`.
- `inspection_records.inspector_id` references `users.user_id`.
- `ai_assessment_results.image_id` references `image_records.image_id`.
- `reports.inspection_id` references `inspection_records.inspection_id`.
- `reports.created_by` references `users.user_id`.
- `audit_logs.user_id` references `users.user_id`.

The recommended default is to prevent accidental deletion of records that are needed for audit history. Use explicit archive or soft-delete fields where appropriate. Cascading deletes should be used only after considering legal retention, reporting, and evidence requirements.

## 11. Database relationship diagram

The following Mermaid ER diagram shows the main planned relationships. It is a logical design and does not create these tables.

```mermaid
erDiagram
    USERS ||--o{ IMAGE_RECORDS : uploads
    USERS ||--o{ INSPECTION_RECORDS : performs
    USERS ||--o{ REPORTS : creates
    USERS ||--o{ AUDIT_LOGS : performs
    INFRASTRUCTURE_ASSETS ||--o{ IMAGE_RECORDS : contains
    INFRASTRUCTURE_ASSETS ||--o{ INSPECTION_RECORDS : receives
    IMAGE_RECORDS ||--o{ AI_ASSESSMENT_RESULTS : produces
    INSPECTION_RECORDS ||--o{ REPORTS : summarizes

    USERS {
        uuid user_id PK
        string name
        string email UK
        string password_hash
        string role
        datetime created_at
        datetime updated_at
    }
    INFRASTRUCTURE_ASSETS {
        uuid asset_id PK
        string asset_name
        string infrastructure_type
        string location
        string description
    }
    IMAGE_RECORDS {
        uuid image_id PK
        uuid asset_id FK
        uuid uploaded_by FK
        string file_path
        datetime upload_date
        string image_status
    }
    INSPECTION_RECORDS {
        uuid inspection_id PK
        uuid asset_id FK
        uuid inspector_id FK
        datetime inspection_date
        string status
    }
    AI_ASSESSMENT_RESULTS {
        uuid result_id PK
        uuid image_id FK
        string model_name
        string predicted_defect_class
        decimal confidence_score
        datetime assessment_date
    }
    REPORTS {
        uuid report_id PK
        uuid inspection_id FK
        string report_path
        datetime creation_date
    }
    AUDIT_LOGS {
        uuid audit_id PK
        uuid user_id FK
        string action
        datetime created_at
    }
```

## 12. Data validation, security, and backup considerations

### Data validation

- Mark required identifiers, names, dates, paths, statuses, and relationships as `NOT NULL` where appropriate.
- Use unique constraints for normalized email addresses and model/version identifiers where needed.
- Use `CHECK` constraints for confidence scores from 0 to 1, positive file sizes, positive dimensions, and controlled status values.
- Validate file content, not only the filename extension, before creating a validated image record.
- Use UTC timestamps and consistent timezone-aware database types.
- Add indexes for foreign keys and common filters such as asset, inspection date, status, and upload date.

### Security

- Never store plain-text passwords. Prefer an external identity provider or use Argon2id/bcrypt through a maintained authentication library.
- Restrict database credentials by environment and role; the application should not use an unrestricted administrator account for normal requests.
- Encrypt database connections and protect object-storage paths with private access controls.
- Enforce ownership and role checks in the backend, not only in the frontend.
- Keep secrets, tokens, and unnecessary personal data out of audit details and reports.
- Use parameterized queries or a trusted ORM to prevent SQL injection.

### Backup and recovery

- Schedule encrypted PostgreSQL backups and test restoration regularly.
- Define retention periods and recovery objectives before production use.
- Back up object storage or configure versioning and lifecycle policies for original images and reports.
- Keep database and object-storage references consistent, and document what happens when one side of a stored file operation fails.
- Record schema changes through versioned migrations rather than manual, untracked edits.

These tables, constraints, security controls, and backup procedures are planned components. They must be implemented, tested, and reviewed before the system is described as production-ready.

## Implementation status boundary

This notebook documents a proposed database design only. It does not create a PostgreSQL database, schema migration, database file, or application integration. The tables and relationships should be treated as planned until implementation and validation are completed.